In [7]:
with open("city_guide.txt", "w") as f:
    f.write("""
Austin is the capital of Texas and known for live music, food trucks, and outdoor activities.

See:
Texas State Capitol, Lady Bird Lake, Barton Springs Pool

Do:
Live music on 6th Street, kayaking, hiking trails

Eat:
BBQ, Tex-Mex, food trucks

Stay safe:
Downtown is safe, avoid isolated areas late night
""")
print("city_guide.txt created successfully")

city_guide.txt created successfully


In [8]:
def retrieve_from_guide(question):
    with open("city_guide.txt", "r") as f:
        text = f.read().lower()

    question = question.lower()

    if "see" in question or "visit" in question:
        return "Texas State Capitol, Lady Bird Lake, Barton Springs Pool"
    if "do" in question:
        return "Live music on 6th Street, kayaking, hiking trails"
    if "eat" in question or "food" in question:
        return "BBQ, Tex-Mex, food trucks"
    if "safe" in question:
        return "Downtown is safe, avoid isolated areas late night"

    return "Not found in guide"


In [9]:
import requests

def geocode_place(place):
    url = "https://nominatim.openstreetmap.org/search"
    params = {
        "q": place,
        "format": "json"
    }
    headers = {
        "User-Agent": "TravelAssistantBot/1.0 (learning project)"
    }

    response = requests.get(url, params=params, headers=headers)


    if response.status_code != 200 or response.text.strip() == "":
        print("Geocode API error")
        return None, None

    try:
        data = response.json()
    except:
        print("Geocode JSON error")
        return None, None

    if len(data) == 0:
        print("No location found")
        return None, None

    return float(data[0]["lat"]), float(data[0]["lon"])



ModuleNotFoundError: No module named 'requests'

In [ ]:
def get_weather(lat, lon):
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "daily": "temperature_2m_max,temperature_2m_min,precipitation_probability_max",
        "timezone": "auto"
    }

    response = requests.get(url, params=params)

    if response.status_code != 200 or response.text.strip() == "":
        return {"error": "Weather API error"}

    try:
        data = response.json()
    except:
        return {"error": "Weather JSON error"}

    if "daily" not in data:
        return {"error": "No weather data"}

    daily = data["daily"]

    return {
        "max_temp": daily["temperature_2m_max"][0],
        "min_temp": daily["temperature_2m_min"][0],
        "rain_prob": daily["precipitation_probability_max"][0]
    }



In [ ]:
def router(user_input):
    text = user_input.lower().strip()


    if "weather" in text or "rain" in text or "temperature" in text:
        return "TOOL_WEATHER"

    if "plan" in text or "trip" in text:
        return "ITINERARY"

    if any(word in text for word in ["food", "eat", "see", "visit", "safe", "do", "famous"]):
        return "RETRIEVE"

    if len(text.split()) <= 3:
        return "TOOL_GEOCODE"

    return "RETRIEVE"



In [ ]:
def create_itinerary(city):
    return {
        "city": city,
        "days": 2,
        "itinerary": [
            {
                "time_block": "Morning",
                "plan": ["Texas State Capitol", "Downtown walk"],
                "notes": ["Sightseeing"]
            },
            {
                "time_block": "Afternoon",
                "plan": ["Food trucks lunch", "Lady Bird Lake kayaking"],
                "notes": ["Outdoor activities"]
            },
            {
                "time_block": "Evening",
                "plan": ["6th Street live music", "BBQ dinner"],
                "notes": ["Nightlife"]
            }
        ],
        "sources": ["city_guide.txt"],
        "tools_used": ["geocode_place", "get_weather"]
    }


In [ ]:
def travel_agent(user_input):
    action = router(user_input)

    if action == "TOOL_WEATHER":
        lat, lon = geocode_place("Austin, TX")
        weather = get_weather(lat, lon)
        return f"Weather Summary: Max {weather['max_temp']}°C, Min {weather['min_temp']}°C, Rain Chance {weather['rain_prob']}%"

    elif action == "ITINERARY":
        return create_itinerary("Austin, TX")

    elif action == "TOOL_GEOCODE":
        lat, lon = geocode_place(user_input)
        return {"city": user_input, "lat": lat, "lon": lon}

    else:
        return retrieve_from_guide(user_input)



In [ ]:
print(travel_agent("What food is famous in Austin?"))


BBQ, Tex-Mex, food trucks


In [ ]:
print(travel_agent("Plan 2 day trip in Austin"))


{'city': 'Austin, TX', 'days': 2, 'itinerary': [{'time_block': 'Morning', 'plan': ['Texas State Capitol', 'Downtown walk'], 'notes': ['Sightseeing']}, {'time_block': 'Afternoon', 'plan': ['Food trucks lunch', 'Lady Bird Lake kayaking'], 'notes': ['Outdoor activities']}, {'time_block': 'Evening', 'plan': ['6th Street live music', 'BBQ dinner'], 'notes': ['Nightlife']}], 'sources': ['city_guide.txt'], 'tools_used': ['geocode_place', 'get_weather']}


In [ ]:
print(travel_agent("Will it rain in Austin?"))


Weather Summary: Max 17.6°C, Min 6.5°C, Rain Chance 0%
